In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
import numpy as np

In [4]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [5]:
path = '/workspaces/crmprtd/update_sheet/'
df = pd.read_excel(path + '20250918-Metadata-AllRequiredChanges.xlsx', header = 1)   # pandas automatically uses openpyxl
df_loc = df[
    (df["ISSUE"].str.strip().str.lower() == "location") &
    (df["Network ID"] != 11)
]
df_loc_new =  df_loc[['Station ID', 'Unique Names','Unique Location (String)', 'Native ID']].reset_index(drop=True)
df_loc_new


,Station ID,Unique Names,Unique Location (String),Native ID
0,12617.0,Martin's Gulch,"123.764438 W, 48.51616 N, Elev. 512.14 m -> 12...",FW007
1,12998.0,ALEX FRASER TOP,"122.94392 W, 49.16187 N, Elev. null m -> 122.9...",14096
2,12999.0,ALEX FRASER CROSS BEAM,"122.94391 W, 49.1618 N, Elev. null m -> 122.94...",14097
3,2733.0,Cayoosh Summit,"122.47403 W, 50.37978 N, Elev. 1350 m -> 122.4...",26224
4,12469.0,Big Bar,"123.700556 W, 48.516111 N, Elev. 180 m -> 121....",28071
...,...,...,...,...
188,2490.0,Parsnip Upper,"122.1458333 W, 54.6125 N, Elev. 790 m -> 122.1...",PAR
189,13124.0,Salmon River Below Campbell Lake Diversion 2,"125.669767 W, 50.093619 N, Elev. 226 m -> 125....",SBD2
190,12409.0,Townsend,"122.24 W, 56.83 N, Elev. 1011 m -> 122.23972 W...",TNS
191,13117.0,Tsaydaychi Lake,"124.096333 W, 55.41635 N, Elev. 1189 m -> 124....",TSY


In [ ]:
import re


def _parse_single_location(loc_str):
    """
    Parse:
    '123.764438 W, 48.51616 N, Elev. 512.14 m'
    '123.764438 W, 48.51616 N, Elev. null m'

    Returns: (lat, lon, elev)
    """
    if pd.isna(loc_str):
        return np.nan, np.nan, np.nan

    pattern = (
        r"([\d\.]+)\s*([EW]),\s*"
        r"([\d\.]+)\s*([NS]),\s*"
        r"Elev\.\s*(null|[\d\.]+)\s*m"
    )

    m = re.search(pattern, loc_str, re.IGNORECASE)
    if not m:
        return np.nan, np.nan, np.nan

    lon_val, lon_dir, lat_val, lat_dir, elev_val = m.groups()

    lon = float(lon_val) * (-1 if lon_dir.upper() == "W" else 1)
    lat = float(lat_val) * (-1 if lat_dir.upper() == "S" else 1)

    elev = np.nan if elev_val.lower() == "null" else float(elev_val)

    return lat, lon, elev


def parse_old_new_lat_lon_elev(loc_str):
    """
    Returns:
    (old_lat, old_lon, old_elev, new_lat, new_lon, new_elev)
    """
    if pd.isna(loc_str):
        return (np.nan,) * 6

    if "->" in loc_str:
        old_str, new_str = map(str.strip, loc_str.split("->", 1))
    else:
        old_str = new_str = loc_str.strip()

    old_lat, old_lon, old_elev = _parse_single_location(old_str)
    new_lat, new_lon, new_elev = _parse_single_location(new_str)

    return (
        old_lat, old_lon, old_elev,
        new_lat, new_lon, new_elev
    )

cols = [
    'old_lat', 'old_lon', 'old_elev',
    'new_lat', 'new_lon', 'new_elev'
]

df_loc_new[cols] = df_loc_new['Unique Location (String)'].apply(
    lambda x: pd.Series(parse_old_new_lat_lon_elev(x))
)
# 
df_loc_new = df_loc_new.drop(columns=['Unique Location (String)'])

In [7]:
df_loc_new

,Station ID,Unique Names,Native ID,old_lat,old_lon,old_elev,new_lat,new_lon,new_elev
0,12617.0,Martin's Gulch,FW007,48.516160,-123.764438,512.14,48.516160,-123.764438,512.0
1,12998.0,ALEX FRASER TOP,14096,49.161870,-122.943920,NaN,49.161870,-122.943920,130.0
2,12999.0,ALEX FRASER CROSS BEAM,14097,49.161800,-122.943910,NaN,49.161800,-122.943910,80.0
3,2733.0,Cayoosh Summit,26224,50.379780,-122.474030,1350.00,50.379780,-122.474030,1250.0
4,12469.0,Big Bar,28071,48.516111,-123.700556,180.00,51.146930,-121.501620,1049.0
...,...,...,...,...,...,...,...,...,...
188,2490.0,Parsnip Upper,PAR,54.612500,-122.145833,790.00,54.605556,-122.137580,790.0
189,13124.0,Salmon River Below Campbell Lake Diversion 2,SBD2,50.093619,-125.669767,226.00,50.093500,-125.669767,226.0
190,12409.0,Townsend,TNS,56.830000,-122.240000,1011.00,56.830000,-122.239720,1011.0
191,13117.0,Tsaydaychi Lake,TSY,55.416350,-124.096333,1189.00,55.416350,-124.760000,1189.0


In [ ]:
check_sql = sa.text("""
SELECT
    station_id,
    lat,
    lon,
    elev
FROM meta_history
WHERE station_id = :station_id
""")


In [ ]:
import numpy as np

with engine.begin() as conn:
    for _, row in df_loc_new.iterrows():

        station_id = int(row["Station ID"])

        old_lat  = row["old_lat"]
        old_lon  = row["old_lon"]
        old_elev = row["old_elev"]

        new_lat  = row["new_lat"]
        new_lon  = row["new_lon"]
        new_elev = row["new_elev"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---
        def norm(x):
            if pd.isna(x):
                return None
            return round(float(x), 6)

        def same(a, b):
            return norm(a) == norm(b)

        if not (
            same(db_row.lat,  old_lat) and
            same(db_row.lon,  old_lon) and
            same(db_row.elev, old_elev)
        ):
            print(
                f"⚠️ Station {station_id} skipped (DB values differ)\n"
                f"   DB : lat={db_row.lat}, lon={db_row.lon}, elev={db_row.elev}\n"
                f"   XLS: lat={old_lat}, lon={old_lon}, elev={old_elev}"
            )
            continue


⚠️ Station 2423 skipped (DB values differ)
   DB : lat=53.23395, lon=-126.1132667, elev=860
   XLS: lat=53.234, lon=-126.113267, elev=860.0
⚠️ Station 2322 skipped (DB values differ)
   DB : lat=50.3689, lon=-126.4336, elev=510
   XLS: lat=49.6673, lon=-115.2144, elev=1051.0


In [13]:
import numpy as np

update_sql = sa.text("""
UPDATE meta_history
SET
    lat  = :new_lat,
    lon  = :new_lon,
    elev = :new_elev
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for _, row in df_loc_new.iterrows():

        station_id = int(row["Station ID"])

        old_lat  = row["old_lat"]
        old_lon  = row["old_lon"]
        old_elev = row["old_elev"]

        new_lat  = row["new_lat"]
        new_lon  = row["new_lon"]
        new_elev = row["new_elev"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 3. Perform update ---
        result = conn.execute(
            update_sql,
            {
                "station_id": station_id,
                "new_lat": new_lat,
                "new_lon": new_lon,
                "new_elev": new_elev,
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {station_id}: "
                f"({old_lat}, {old_lon}, {old_elev}) → "
                f"({new_lat}, {new_lon}, {new_elev})"
            )
        else:
            print(f"⚠️ Station {station_id}: no update applied")

✅ Updated station 12617: (48.51616, -123.764438, 512.14) → (48.51616, -123.764438, 512.0)
✅ Updated station 12998: (49.16187, -122.94392, nan) → (49.16187, -122.94392, 130.0)
✅ Updated station 12999: (49.1618, -122.94391, nan) → (49.1618, -122.94391, 80.0)
✅ Updated station 2733: (50.37978, -122.47403, 1350.0) → (50.37978, -122.47403, 1250.0)
✅ Updated station 12469: (48.516111, -123.700556, 180.0) → (51.14693, -121.50162, 1049.0)
✅ Updated station 12510: (48.516111, -123.700556, 180.0) → (51.1929, -121.5295, 1093.0)
✅ Updated station 2761: (49.75778, -119.12639, 1220.0) → (49.75778, -119.12639, 1200.0)
✅ Updated station 2771: (50.04631, -117.17914, 1080.0) → (50.04631, -117.17914, 1070.0)
✅ Updated station 13027: (49.37254, -115.04689, nan) → (49.37254, -115.04689, 1998.0)
✅ Updated station 2800: (51.27303, -116.76139, 1140.0) → (51.27483, -116.76653, 1140.0)
✅ Updated station 12988: (51.02732, -116.53516, nan) → (51.02732, -116.53516, 797.0)
✅ Updated station 12997: (50.43048, -116.3

In [15]:
import numpy as np

with engine.begin() as conn:
    for _, row in df_loc_new.iterrows():

        station_id = int(row["Station ID"])

        new_lat  = row["new_lat"]
        new_lon  = row["new_lon"]
        new_elev = row["new_elev"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---
        def norm(x):
            if pd.isna(x):
                return None
            return round(float(x), 6)

        def same(a, b):
            return norm(a) == norm(b)

        if not (
            same(db_row.lat,  new_lat) and
            same(db_row.lon,  new_lon) and
            same(db_row.elev, new_elev)
        ):
            print(
                f"⚠️ Station {station_id} skipped (DB values differ)\n"
                f"   DB : lat={db_row.lat}, lon={db_row.lon}, elev={db_row.elev}\n"
                f"   XLS: lat={new_lat}, lon={new_lon}, elev={new_elev}"
            )
        else:
            print(
                f"✅ Station {station_id} already matches new values "
                f"(lat={db_row.lat}, lon={db_row.lon}, elev={db_row.elev})"
            )


✅ Station 12617 already matches new values (lat=48.51616, lon=-123.764438, elev=512.0)
✅ Station 12998 already matches new values (lat=49.16187, lon=-122.94392, elev=130.0)
✅ Station 12999 already matches new values (lat=49.1618, lon=-122.94391, elev=80.0)
✅ Station 2733 already matches new values (lat=50.37978, lon=-122.47403, elev=1250.0)
✅ Station 12469 already matches new values (lat=51.14693, lon=-121.50162, elev=1049.0)
✅ Station 12510 already matches new values (lat=51.1929, lon=-121.5295, elev=1093.0)
✅ Station 2761 already matches new values (lat=49.75778, lon=-119.12639, elev=1200.0)
✅ Station 2771 already matches new values (lat=50.04631, lon=-117.17914, elev=1070.0)
✅ Station 13027 already matches new values (lat=49.37254, lon=-115.04689, elev=1998.0)
✅ Station 2800 already matches new values (lat=51.27483, lon=-116.76653, elev=1140.0)
✅ Station 12988 already matches new values (lat=51.02732, lon=-116.53516, elev=797.0)
✅ Station 12997 already matches new values (lat=50.430